In [2]:
from dash import Dash, dcc, html, dash_table
import dash_leaflet as dl
import plotly.express as px
from dash.dependencies import Input, Output
import base64
import pandas as pd

from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# MongoDB authentication
username = "aacuser"
password = "aacpass"

db = AnimalShelter(username, password)

# Load the complete dataset for the Reset view
df = pd.DataFrame.from_records(db.read({}))

# MongoDB ObjectId values are not compatible with the Dash DataTable
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


# Rescue criteria from the original Grazioso Salvare requirements
RESCUE_CRITERIA = {
    "water": {
        "label": "Water Rescue",
        "breeds": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ],
        "sex": "Intact Female",
        "min_age": 26,
        "max_age": 156
    },

    "mountain": {
        "label": "Mountain or Wilderness Rescue",
        "breeds": [
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler"
        ],
        "sex": "Intact Male",
        "min_age": 26,
        "max_age": 156
    },

    "disaster": {
        "label": "Disaster or Individual Tracking",
        "breeds": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler"
        ],
        "sex": "Intact Male",
        "min_age": 20,
        "max_age": 300
    }
}


def build_rescue_pipeline(criteria):
    """
    Build a MongoDB aggregation pipeline that identifies animals from the
    preferred breed group, scores each candidate using the original rescue
    criteria, ranks the results, and calculates summary statistics.
    """

    return [
        {
            "$match": {
                "animal_type": "Dog",
                "breed": {
                    "$in": criteria["breeds"]
                }
            }
        },

        {
            "$addFields": {
                # Breed is the entry requirement for the candidate pool.
                "breed_match": 1,

                # Award one point when the preferred sex matches.
                "sex_match": {
                    "$cond": [
                        {
                            "$eq": [
                                "$sex_upon_outcome",
                                criteria["sex"]
                            ]
                        },
                        1,
                        0
                    ]
                },

                # Award one point when the animal is within the preferred
                # training-age range.
                "age_match": {
                    "$cond": [
                        {
                            "$and": [
                                {
                                    "$gte": [
                                        "$age_upon_outcome_in_weeks",
                                        criteria["min_age"]
                                    ]
                                },
                                {
                                    "$lte": [
                                        "$age_upon_outcome_in_weeks",
                                        criteria["max_age"]
                                    ]
                                }
                            ]
                        },
                        1,
                        0
                    ]
                }
            }
        },

        {
            "$addFields": {
                # Candidate scores range from 1 to 3.
                "candidate_score": {
                    "$add": [
                        "$breed_match",
                        "$sex_match",
                        "$age_match"
                    ]
                }
            }
        },

        {
            "$addFields": {
                # Convert the numeric score into a user-friendly label.
                "match_strength": {
                    "$switch": {
                        "branches": [
                            {
                                "case": {
                                    "$eq": [
                                        "$candidate_score",
                                        3
                                    ]
                                },
                                "then": "Strong Match"
                            },
                            {
                                "case": {
                                    "$eq": [
                                        "$candidate_score",
                                        2
                                    ]
                                },
                                "then": "Moderate Match"
                            }
                        ],
                        "default": "Basic Match"
                    }
                }
            }
        },

        {
            "$facet": {
                # Return the ranked candidate records.
                "candidates": [
                    {
                        "$sort": {
                            "candidate_score": -1,
                            "breed": 1,
                            "age_upon_outcome_in_weeks": 1
                        }
                    },

                    # Internal scoring fields are not needed by the user.
                    {
                        "$project": {
                            "breed_match": 0,
                            "sex_match": 0,
                            "age_match": 0
                        }
                    }
                ],

                # Calculate summary information from the same candidate pool.
                "summary": [
                    {
                        "$group": {
                            "_id": None,

                            "total_candidates": {
                                "$sum": 1
                            },

                            "strong_matches": {
                                "$sum": {
                                    "$cond": [
                                        {
                                            "$eq": [
                                                "$candidate_score",
                                                3
                                            ]
                                        },
                                        1,
                                        0
                                    ]
                                }
                            },

                            "moderate_matches": {
                                "$sum": {
                                    "$cond": [
                                        {
                                            "$eq": [
                                                "$candidate_score",
                                                2
                                            ]
                                        },
                                        1,
                                        0
                                    ]
                                }
                            },

                            "basic_matches": {
                                "$sum": {
                                    "$cond": [
                                        {
                                            "$eq": [
                                                "$candidate_score",
                                                1
                                            ]
                                        },
                                        1,
                                        0
                                    ]
                                }
                            },

                            "average_score": {
                                "$avg": "$candidate_score"
                            }
                        }
                    }
                ]
            }
        }
    ]


#########################
# Dashboard Layout / View
#########################

app = Dash(__name__)

# Load Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'

encoded_image = base64.b64encode(
    open(image_filename, 'rb').read()
).decode()


# Standard columns used by the original dataset
base_columns = [
    {
        "name": column,
        "id": column,
        "deletable": False,
        "selectable": True
    }
    for column in df.columns
]


# Enhanced rescue views display the new decision-support fields first
scored_columns = [
    {
        "name": "Candidate Score",
        "id": "candidate_score",
        "deletable": False,
        "selectable": True
    },

    {
        "name": "Match Strength",
        "id": "match_strength",
        "deletable": False,
        "selectable": True
    }
] + base_columns


app.layout = html.Div([

    # Header
    html.Center([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image),
                style={
                    'height': '150px'
                }
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),

        html.H1(
            'Grazioso Salvare Dashboard'
        ),

        html.H3(
            'Tyler Shurilla'
        )
    ]),

    html.Hr(),

    # Rescue selection
    html.Div([
        dcc.RadioItems(
            id='filter-type',

            options=[
                {
                    'label': 'Reset',
                    'value': 'reset'
                },
                {
                    'label': 'Water Rescue',
                    'value': 'water'
                },
                {
                    'label': 'Mountain or Wilderness Rescue',
                    'value': 'mountain'
                },
                {
                    'label': 'Disaster or Individual Tracking',
                    'value': 'disaster'
                }
            ],

            value='reset',
            inline=True
        )
    ], style={
        'marginBottom': '15px'
    }),

    # Explanation of the enhanced scoring system
    html.Div([
        html.Strong(
            'Rescue Candidate Scoring: '
        ),

        html.Span(
            '3 = Strong Match, 2 = Moderate Match, '
            '1 = Basic Match. Candidates must first belong '
            'to a preferred breed group. Additional points '
            'are awarded for matching the preferred sex and '
            'training-age range.'
        )
    ], style={
        'marginBottom': '15px'
    }),

    html.Hr(),

    # Database-generated summary information
    html.Div(
        id='summary-id',
        style={
            'marginBottom': '20px'
        }
    ),

    # Animal records
    dash_table.DataTable(
        id='datatable-id',

        columns=base_columns,

        data=df.to_dict('records'),

        row_selectable='single',

        selected_rows=[0],

        page_size=10,

        sort_action='native',

        filter_action='native',

        style_table={
            'overflowX': 'auto'
        },

        style_cell={
            'textAlign': 'left',
            'padding': '5px',
            'minWidth': '120px',
            'width': '120px',
            'maxWidth': '180px',
            'whiteSpace': 'normal'
        },

        style_header={
            'backgroundColor': 'rgb(230, 230, 230)',
            'fontWeight': 'bold'
        }
    ),

    html.Br(),
    html.Hr(),

    # Chart and geolocation map
    html.Div(
        className='row',

        style={
            'display': 'flex'
        },

        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6'
            ),

            html.Div(
                id='map-id',
                className='col s12 m6'
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    [
        Output(
            'datatable-id',
            'data'
        ),

        Output(
            'datatable-id',
            'columns'
        ),

        Output(
            'datatable-id',
            'selected_rows'
        ),

        Output(
            'summary-id',
            'children'
        )
    ],

    Input(
        'filter-type',
        'value'
    )
)
def update_dashboard(filter_type):

    # Reset keeps the original dashboard behavior and shows all records.
    if filter_type == 'reset':

        summary = html.Div([
            html.H3(
                'All Shelter Records'
            ),

            html.P(
                f'{len(df):,} records are currently available. '
                'Select a rescue type to score and rank candidates.'
            )
        ])

        return (
            df.to_dict('records'),
            base_columns,
            [0],
            summary
        )

    # Get the criteria associated with the selected rescue type.
    criteria = RESCUE_CRITERIA.get(filter_type)

    if criteria is None:
        return (
            df.to_dict('records'),
            base_columns,
            [0],
            html.P(
                'Unable to determine the selected rescue type.'
            )
        )

    # Execute the MongoDB aggregation pipeline.
    pipeline = build_rescue_pipeline(criteria)

    aggregation_results = db.aggregate(pipeline)

    if not aggregation_results:
        return (
            [],
            scored_columns,
            [],
            html.P(
                'No rescue candidates were found.'
            )
        )

    result = aggregation_results[0]

    candidates = result.get(
        'candidates',
        []
    )

    summary_results = result.get(
        'summary',
        []
    )

    # ObjectId values cannot be sent to the Dash DataTable.
    for animal in candidates:
        animal.pop(
            '_id',
            None
        )

    # Handle an empty summary safely.
    if summary_results:
        summary_data = summary_results[0]
    else:
        summary_data = {
            'total_candidates': 0,
            'strong_matches': 0,
            'moderate_matches': 0,
            'basic_matches': 0,
            'average_score': 0
        }

    average_score = summary_data.get(
        'average_score',
        0
    )

    if average_score is None:
        average_score = 0

    # Display the MongoDB-generated summary.
    summary = html.Div([

        html.H3(
            f'{criteria["label"]} Candidate Summary'
        ),

        html.P([
            html.Strong(
                'Total Candidates: '
            ),
            f'{summary_data.get("total_candidates", 0)}   |   ',

            html.Strong(
                'Strong Matches: '
            ),
            f'{summary_data.get("strong_matches", 0)}   |   ',

            html.Strong(
                'Moderate Matches: '
            ),
            f'{summary_data.get("moderate_matches", 0)}   |   ',

            html.Strong(
                'Basic Matches: '
            ),
            f'{summary_data.get("basic_matches", 0)}   |   ',

            html.Strong(
                'Average Score: '
            ),
            f'{average_score:.2f} / 3'
        ])
    ])

    return (
        candidates,
        scored_columns,
        [0] if candidates else [],
        summary
    )


# Build the breed-distribution chart from the currently displayed records.
@app.callback(
    Output(
        'graph-id',
        'children'
    ),

    Input(
        'datatable-id',
        'derived_virtual_data'
    )
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(
        viewData
    )

    if 'breed' not in dff.columns:
        return []

    breed_counts = (
        dff['breed']
        .value_counts()
        .reset_index()
    )

    breed_counts.columns = [
        'breed',
        'count'
    ]

    # Group smaller categories into Other to keep the chart readable.
    if len(breed_counts) > 10:

        top_breeds = (
            breed_counts
            .head(10)
            .copy()
        )

        other_count = (
            breed_counts['count']
            .iloc[10:]
            .sum()
        )

        other_row = pd.DataFrame([
            {
                'breed': 'Other',
                'count': other_count
            }
        ])

        chart_data = pd.concat(
            [
                top_breeds,
                other_row
            ],
            ignore_index=True
        )

    else:
        chart_data = breed_counts

    return [
        dcc.Graph(
            figure=px.pie(
                chart_data,
                names='breed',
                values='count',
                title='Breed Distribution'
            )
        )
    ]


# Highlight the selected row.
@app.callback(
    Output(
        'datatable-id',
        'style_data_conditional'
    ),

    Input(
        'datatable-id',
        'selected_rows'
    )
)
def update_styles(selected_rows):

    if selected_rows is None or len(selected_rows) == 0:
        return []

    return [
        {
            'if': {
                'row_index': selected_rows[0]
            },

            'backgroundColor': '#D2F3FF'
        }
    ]


# Update the map using named fields instead of column positions.
@app.callback(
    Output(
        'map-id',
        'children'
    ),

    [
        Input(
            'datatable-id',
            'derived_virtual_data'
        ),

        Input(
            'datatable-id',
            'derived_virtual_selected_rows'
        )
    ]
)
def update_map(viewData, index):

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(
        viewData
    )

    # Default to the first visible row.
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Protect against a stale selected-row index after a filter change.
    if row >= len(dff):
        row = 0

    selected_animal = dff.iloc[row]

    latitude = selected_animal.get(
        'location_lat'
    )

    longitude = selected_animal.get(
        'location_long'
    )

    if pd.isna(latitude) or pd.isna(longitude):
        return [
            html.P(
                'Geolocation information is not available for this animal.'
            )
        ]

    animal_name = selected_animal.get(
        'name'
    )

    if pd.isna(animal_name) or not animal_name:
        animal_name = 'Unnamed Animal'

    breed = selected_animal.get(
        'breed',
        'Unknown Breed'
    )

    return [
        dl.Map(
            style={
                'width': '1000px',
                'height': '500px'
            },

            center=[
                latitude,
                longitude
            ],

            zoom=10,

            children=[
                dl.TileLayer(
                    id='base-layer-id'
                ),

                dl.Marker(
                    position=[
                        latitude,
                        longitude
                    ],

                    children=[
                        dl.Tooltip(
                            breed
                        ),

                        dl.Popup([
                            html.H3(
                                animal_name
                            ),

                            html.P(
                                breed
                            )
                        ])
                    ]
                )
            ]
        )
    ]


#########################
# Run Dashboard
#########################

app.run(
    jupyter_mode='external',
    debug=False
)

Dash app running on http://127.0.0.1:8050/
